In [7]:
from dotenv import load_dotenv
import json
import os
from pathlib import Path
from pydub import AudioSegment
import shutil, errno
import pandas as pd
from elevenlabs.client import ElevenLabs
from elevenlabs import play
from io import BytesIO
from elevenlabs.client import ElevenLabs

In [8]:
def extension(file):
    return Path(file).suffix.replace('.', '')

In [9]:
load_dotenv('openai.env')

True

In [10]:
file = 'kate-sample.m4a'
output_name = Path(file).stem.replace(' ', '_')

In [11]:
elevenlabs = ElevenLabs(
  api_key=os.getenv("ELEVENLABS_API_KEY"),
)

In [12]:
with open(file, 'rb') as in_file:
    raw_audio = BytesIO(in_file.read())

In [13]:
transcription = elevenlabs.speech_to_text.convert(
    file=raw_audio,
    model_id="scribe_v1", # Model to use, for now only "scribe_v1" is supported
    tag_audio_events=True, # Tag audio events like laughter, applause, etc.
    language_code="eng", # Language of the audio file. If set to None, the model will detect the language automatically.
    diarize=True, # Whether to annotate who is speaking
)

In [14]:
transcript = dict(transcription)

In [16]:
transcript.keys()

dict_keys(['language_code', 'language_probability', 'text', 'words', 'additional_formats'])

In [17]:
result = []
i = 0
prev_speaker = dict(transcript['words'][0])['speaker_id']
result.append({
            'speaker_id': prev_speaker,
            'text': '',
            'words': []
        })
for word in transcript['words']:
    word = dict(word)
    curr_speaker = word['speaker_id']
    if curr_speaker == prev_speaker:
        result[i]['text'] += word['text']
        result[i]['words'].append(word)
    else:
        i += 1
        result.append({
            'speaker_id': curr_speaker,
            'text': word['text'],
            'words': [word]
        })
    prev_speaker = curr_speaker

In [53]:
result = {
    'transcript':{
        'segments': result
    }
}

In [54]:
with open(f'{output_name}_transcript.json', 'w') as out_path:  
    json.dump(result, out_path)